# Air Pollution Exposure Assessment for Health Studies

**Goal:** Estimate pollution exposure at study locations (GP surgeries in London)
by finding the nearest monitors, downloading their data, and assigning
exposure estimates — a workflow common in environmental epidemiology.

**API keys required:** `BL_API_KEY` (Breathe London) for enhanced spatial coverage

**Aeolus features demonstrated:**
- `find_sites()` with `near` parameter for each study location
- Multi-source metadata aggregation (AURN + Breathe London)
- `download()` with dict format for cross-network data
- `metrics.time_average()` for seasonal/annual aggregation
- Geospatial distance calculations via `aeolus.geo`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Breathe London enhances spatial coverage but isn't strictly required
has_bl_key = bool(os.environ.get("BL_API_KEY"))
if not has_bl_key:
    print("BL_API_KEY not set \u2014 running with AURN only (reduced spatial coverage).")
    print("Get a key at https://www.breathelondon.org/developers")

In [ ]:
import aeolus
from aeolus import metrics
from aeolus.geo import haversine_distance
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Define Study Locations

We define 10 hypothetical GP surgery locations across London.
In a real study, these would come from your cohort database.

In [ ]:
# Study locations (GP surgeries across London boroughs)
study_locations = pd.DataFrame([
    {"id": "GP01", "name": "Camden Medical Centre",      "lat": 51.5392, "lon": -0.1426},
    {"id": "GP02", "name": "Hackney Health Centre",      "lat": 51.5450, "lon": -0.0553},
    {"id": "GP03", "name": "Southwark Surgery",          "lat": 51.5034, "lon": -0.0887},
    {"id": "GP04", "name": "Tower Hamlets Practice",     "lat": 51.5150, "lon": -0.0340},
    {"id": "GP05", "name": "Westminster Clinic",         "lat": 51.4975, "lon": -0.1357},
    {"id": "GP06", "name": "Greenwich Surgery",          "lat": 51.4769, "lon": 0.0005},
    {"id": "GP07", "name": "Lewisham Health Centre",     "lat": 51.4535, "lon": -0.0205},
    {"id": "GP08", "name": "Lambeth Practice",           "lat": 51.4571, "lon": -0.1231},
    {"id": "GP09", "name": "Islington Surgery",          "lat": 51.5362, "lon": -0.1033},
    {"id": "GP10", "name": "Wandsworth Medical Centre",  "lat": 51.4571, "lon": -0.1910},
])

print(f"{len(study_locations)} study locations defined")
study_locations

## 2. Find Nearest Monitors for Each Location

For each GP surgery, find the closest air quality monitor(s). We search
AURN (and Breathe London if available) to maximise spatial coverage.

In [ ]:
# Build list of sources to search
sources = ["AURN", "AQE"]
if has_bl_key:
    sources.append("BREATHE_LONDON")

# Find nearest monitor for each study location
assignments = []

for _, loc in study_locations.iterrows():
    try:
        nearby = aeolus.find_sites(
            sources,
            near=(float(loc["lat"]), float(loc["lon"])),
            radius_km=5,
        )
    except TypeError:
        # Fallback: some sources may return string coordinates
        nearby = aeolus.find_sites(
            ["AURN"],
            near=(float(loc["lat"]), float(loc["lon"])),
            radius_km=5,
        )
    
    # Ensure lat/lon are numeric in results
    if not nearby.empty:
        nearby["latitude"] = pd.to_numeric(nearby["latitude"], errors="coerce")
        nearby["longitude"] = pd.to_numeric(nearby["longitude"], errors="coerce")
        nearby = nearby.dropna(subset=["latitude", "longitude"])

    if not nearby.empty:
        nearest = nearby.iloc[0]  # Already sorted by distance
        assignments.append({
            "study_id": loc["id"],
            "study_name": loc["name"],
            "monitor_code": nearest["site_code"],
            "monitor_name": nearest["site_name"],
            "network": nearest["source_network"],
            "distance_km": nearest["distance_km"],
        })
    else:
        print(f"\u26a0\ufe0f No monitor within 5km of {loc['name']}")

assign_df = pd.DataFrame(assignments)
print(f"\nAssigned {len(assign_df)} of {len(study_locations)} locations")
assign_df

In [ ]:
# Summarise monitor distances
print(f"Distance statistics:")
print(f"  Mean:   {assign_df['distance_km'].mean():.1f} km")
print(f"  Max:    {assign_df['distance_km'].max():.1f} km")
print(f"  Median: {assign_df['distance_km'].median():.1f} km")

## 3. Download Monitor Data

Download a full year's data from all assigned monitors.

In [ ]:
# Build download map from assignments
download_map = (
    assign_df
    .groupby("network")["monitor_code"]
    .apply(lambda x: x.unique().tolist())
    .to_dict()
)

for network, sites in download_map.items():
    print(f"{network}: {sites}")

# Download one year
data = aeolus.download(
    download_map,
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2024, 12, 31),
)

print(f"\nDownloaded {len(data):,} rows from {data['site_code'].nunique()} monitors")

## 4. Calculate Exposure Estimates

Compute annual and seasonal means per monitor, then assign these as
exposure estimates to each study location.

In [ ]:
# Focus on NO2 (the primary traffic-related pollutant)
no2 = data[data["measurand"] == "NO2"]

# Annual regulatory statistics per monitor
monitor_stats = metrics.aq_stats(no2, pollutant="NO2")

# Map monitor stats to study locations
exposure = assign_df.merge(
    monitor_stats[["site_code", "annual_mean", "data_capture"]],
    left_on="monitor_code",
    right_on="site_code",
    how="left",
).drop(columns="site_code")

print("Exposure estimates (annual mean NO\u2082):")
exposure[["study_id", "study_name", "monitor_name",
          "distance_km", "annual_mean", "data_capture"]]

In [ ]:
# Seasonal means using time_average
quarterly = metrics.time_average(no2, freq="QS")

# Add season labels
season_map = {1: "Winter", 4: "Spring", 7: "Summer", 10: "Autumn"}
quarterly["season"] = quarterly["date_time"].dt.month.map(season_map)

seasonal_means = (
    quarterly[quarterly["measurand"] == "NO2"]
    .groupby(["site_code", "season"])["value"]
    .mean()
    .reset_index()
    .pivot(index="site_code", columns="season", values="value")
)

print("Seasonal NO\u2082 means by monitor (\u00b5g/m\u00b3):")
seasonal_means.round(1)

## 5. Exposure Distribution

Summarise the exposure distribution across the study population.

In [ ]:
valid = exposure.dropna(subset=["annual_mean"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of exposure estimates
axes[0].hist(valid["annual_mean"], bins=10, color="#3498db", alpha=0.7, edgecolor="white")
axes[0].axvline(40, color="red", linestyle="--", label="UK limit (40 \u00b5g/m\u00b3)")
axes[0].axvline(10, color="orange", linestyle="--", label="WHO AQG (10 \u00b5g/m\u00b3)")
axes[0].set_xlabel("Annual mean NO\u2082 (\u00b5g/m\u00b3)")
axes[0].set_ylabel("Number of study locations")
axes[0].set_title("Exposure Distribution")
axes[0].legend(fontsize=8)

# Exposure vs distance from monitor
axes[1].scatter(valid["distance_km"], valid["annual_mean"],
                s=80, color="#e74c3c", alpha=0.7)
for _, row in valid.iterrows():
    axes[1].annotate(row["study_id"], (row["distance_km"], row["annual_mean"]),
                     fontsize=8, ha="left", va="bottom")
axes[1].set_xlabel("Distance to nearest monitor (km)")
axes[1].set_ylabel("Annual mean NO\u2082 (\u00b5g/m\u00b3)")
axes[1].set_title("Exposure vs Monitor Distance")

plt.tight_layout()
plt.show()

print(f"\nExposure summary:")
print(f"  Mean:   {valid['annual_mean'].mean():.1f} \u00b5g/m\u00b3")
print(f"  Range:  {valid['annual_mean'].min():.1f}\u2013{valid['annual_mean'].max():.1f} \u00b5g/m\u00b3")
print(f"  Above WHO guideline (10 \u00b5g/m\u00b3): {(valid['annual_mean'] > 10).sum()} of {len(valid)}")

## Summary

This notebook demonstrated an epidemiological exposure assessment workflow:

1. **Study location definition** — arbitrary coordinates (GP surgeries)
2. **Spatial monitor assignment** — `find_sites(near=...)` for each location
3. **Multi-source download** — AURN + Breathe London for dense coverage
4. **Composable aggregation** — `aq_stats()` and `time_average()` for annual/seasonal means
5. **Exposure summary** — distribution and distance-quality assessment

### Limitations
- Nearest-monitor assignment assumes spatial homogeneity within the monitor's area
- For better estimates, consider inverse-distance weighting from multiple monitors
- Land-use regression or dispersion models provide finer spatial resolution

### Next steps
- Add PM\u2082.\u2085 exposure alongside NO\u2082
- Implement inverse-distance weighting from the 3 nearest monitors
- Export exposure estimates to CSV for linkage with health data